In [10]:
import scipy.io
import pandas as pd
import sys
import os
sys.path.append(os.path.abspath('../..'))
from utlis.sync_utlis.sync_df_utlis import find_min_frame


def align_frames(calib_file, light_change_frames, save_path):
    """
    Aligns camera frames based on the given light change frames.
    
    Parameters:
    - light_change_frames: A dictionary where keys are camera names and values are the frames where light change is detected.
    - camera_data_frames: A dictionary where keys are camera names and values are the corresponding data frames.
    
    Returns:
    - A dictionary with the adjusted data frames.
    """
    calib_data = scipy.io.loadmat(calib_file)
    sync = calib_data['sync']
    # import pdb
    # pdb.set_trace()



    camera_keys = list(light_change_frames.keys())

    processed_drop_frames = {}

    for cam_key in camera_keys:
        drop_frames = light_change_frames[cam_key]
        
        if len(drop_frames) > 1:
            # Get the difference between consecutive frames
            dif = abs(drop_frames[1] - drop_frames[0])
            
            if dif < 2:
                # If the difference is small, use the second frame as the drop frame
                processed_drop_frames[cam_key] = drop_frames[1]
            else:
                # If there is more than one drop frame and the difference is large, raise a warning
                print(f"Warning: Lighting detected for more than 2 frames in {cam_key}, needs specific attention.")
                processed_drop_frames[cam_key] = min(drop_frames)  # Choose the minimum frame as a fallback
        else:
            # If there is only one drop frame, use that one
            processed_drop_frames[cam_key] = drop_frames[0]

    # Print or return the processed drop frames
    print("Processed drop frames:", processed_drop_frames)

                
    min_frame = find_min_frame(processed_drop_frames)
    print(min_frame)

    # adjusted_data_frames = {}
    for cam_key in camera_keys:

        cam_idx = camera_keys.index(cam_key)
        keyyyyy = 'data_frame'  # Assuming this is the key for data frames in your structure
        # print(1)
        # import pdb
        # pdb.set_trace()

        # print(sync[cam_idx][0][keyyyyy])
        data_frame = sync[cam_idx][0][keyyyyy][0][0][0]
        # print(2)
        # print(data_frame)
        # print(sync[cam_idx][0][keyyyyy])

        # Convert to DataFrame for easier manipulation
        df = pd.DataFrame(data_frame)
        # print(3)
        # Adjust frames
        frames_to_remove = processed_drop_frames[cam_key] - min_frame #[0], okay now not just takikng the first thing after some procesing, very cool
        if frames_to_remove > 0:
            df = df.iloc[frames_to_remove:].reset_index(drop=True)
        # print(4)
        # print(df.to_numpy().flatten())
        # adjusted_data_frames[cam_key] = df
        # sync[cam_idx][0][keyyyyy][0][0][0] = None
        sync[cam_idx][0][keyyyyy][0][0] = [df.to_numpy().flatten()]
        # print(5)
        # print(sync[cam_idx][0][keyyyyy])
    # return sync[cam_idx][0][keyyyyy][0][0]
    calib_data['sync'] = sync
    scipy.io.savemat(save_path, calib_data)
    print('alined data saved to:', save_path)

def align_frames_new(calib_file, light_change_frames, save_path):
    """
    Aligns camera frames based on the given light change frames.
    
    Parameters:
    - light_change_frames: A dictionary where keys are camera names and values are the frames where light change is detected.
    - camera_data_frames: A dictionary where keys are camera names and values are the corresponding data frames.
    
    Returns:
    - A dictionary with the adjusted data frames.
    """
    calib_data = scipy.io.loadmat(calib_file)
    sync = calib_data['sync']
    # import pdb
    # pdb.set_trace()

    camera_keys = list(light_change_frames.keys())

    processed_drop_frames = {}

    for cam_key in camera_keys:
        drop_frames = light_change_frames[cam_key]
        
        if len(drop_frames) > 1:
            # Get the difference between consecutive frames
            dif = abs(drop_frames[1] - drop_frames[0])
            
            if dif < 2:
                # If the difference is small, use the second frame as the drop frame
                processed_drop_frames[cam_key] = drop_frames[1]
            else:
                # If there is more than one drop frame and the difference is large, raise a warning
                print(f"Warning: Lighting detected for more than 2 frames in {cam_key}, needs specific attention.")
                processed_drop_frames[cam_key] = min(drop_frames)  # Choose the minimum frame as a fallback
        else:
            # If there is only one drop frame, use that one
            processed_drop_frames[cam_key] = drop_frames[0]

    # Print or return the processed drop frames
    print("Processed drop frames:", processed_drop_frames)

    min_frame = find_min_frame(processed_drop_frames)
    print(min_frame)

    # ===================== (1) 仅新增这一行：计算公共尾部长度 =====================
    target_len = min(len(sync[i][0]['data_frame'][0][0][0]) - (processed_drop_frames[camera_keys[i]] - min_frame) for i in range(len(camera_keys)))

    # adjusted_data_frames = {}
    for cam_key in camera_keys:

        cam_idx = camera_keys.index(cam_key)
        keyyyyy = 'data_frame'  # Assuming this is the key for data frames in your structure

        data_frame = sync[cam_idx][0][keyyyyy][0][0][0]

        # Convert to DataFrame for easier manipulation
        df = pd.DataFrame(data_frame)

        # Adjust frames
        frames_to_remove = processed_drop_frames[cam_key] - min_frame  # [0], okay now not just takikng the first thing after some procesing, very cool
        if frames_to_remove > 0:
            df = df.iloc[frames_to_remove:].reset_index(drop=True)

        # ===================== (2) 仅新增这一行：按公共长度裁尾并重置索引 =====================
        df = df.iloc[:target_len].reset_index(drop=True)

        # ===================== (3) 仅新增这一行：同步裁剪 data_sampleID 的结尾 =====================
        sync[cam_idx][0]['data_sampleID'][0][0] = [sync[cam_idx][0]['data_sampleID'][0][0][0][:target_len]]

        # 写回 data_frame
        sync[cam_idx][0][keyyyyy][0][0] = [df.to_numpy().flatten()]

    calib_data['sync'] = sync
    scipy.io.savemat(save_path, calib_data)
    print('alined data saved to:', save_path)



In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../'))

from utlis.sync_utlis.sync_df_utlis import process_sync
from utlis.exe_engine_utlis.comb_all_exe import mir_generate_param_z

test_path = "/hpc/group/tdunn/Bryan_Rigs/BigOpenField/yankun_calibration_free/2025_10_17/record_mouse_validate_3"

base_path = os.path.dirname(test_path.rstrip("/"))   # .../2025_09_19
rec       = os.path.basename(test_path.rstrip("/"))  # 1eNpHR_10base10opto_12_17
calib     = os.path.join(base_path, "after_mouse_validate")
out = f"{os.path.basename(base_path)}_{rec}_{os.path.basename(calib)}_label3d_dannce.mat"

In [ ]:
process_sync(test_path, threshold=2, max_frames=600, min_frame=250)